In [48]:
import psycopg

print("Psycopg version:", psycopg.__version__)


Psycopg version: 3.3.8


In [80]:
import pandas as pd
import pgserver
from pathlib import Path
from sqlalchemy import create_engine, text

pg = pgserver.get_server("/tmp/hw7_pg", cleanup_mode="stop")

engine = create_engine(pg.get_uri(), future=True)

with engine.connect() as conn:
    print(conn.execute(text("SELECT version()")).scalar_one())

parquet_path = Path("data/yellow_tripdata_2024-03.parquet")

print("File exists:", parquet_path.exists())


PostgreSQL 16.2 on aarch64-apple-darwin23.5.0, compiled by Apple clang version 15.0.0 (clang-1500.0.40.1), 64-bit
File exists: True


In [82]:
columns = [
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "PULocationID",
    "DOLocationID",
    "trip_distance",
    "fare_amount",
    "passenger_count",
]

df = pd.read_parquet(parquet_path, columns=columns).head(500_000)

print("Shape:", df.shape)

display(df.head())


Shape: (500000, 7)


,tpep_pickup_datetime,tpep_dropoff_datetime,PULocationID,DOLocationID,trip_distance,fare_amount,passenger_count
0,2024-03-01 00:18:51,2024-03-01 00:23:45,142,239,1.30,8.6,0.0
1,2024-03-01 00:26:00,2024-03-01 00:29:06,238,24,1.10,7.2,0.0
2,2024-03-01 00:09:22,2024-03-01 00:15:24,263,75,0.86,7.9,1.0
3,2024-03-01 00:33:45,2024-03-01 00:39:34,164,162,0.82,7.9,1.0
4,2024-03-01 00:05:43,2024-03-01 00:26:22,263,7,4.90,25.4,0.0


In [83]:
# Залишаємо коректні суми оплати
df = df[(df["fare_amount"] > 0) & (df["fare_amount"] < 500)]

# Залишаємо коректні відстані
df = df[(df["trip_distance"] > 0) & (df["trip_distance"] < 100)]

# Видаляємо записи без pickup zone
df = df[df["PULocationID"].notna()]

print("Rows after cleaning:", len(df))

display(df.head())


Rows after cleaning: 486964


,tpep_pickup_datetime,tpep_dropoff_datetime,PULocationID,DOLocationID,trip_distance,fare_amount,passenger_count
0,2024-03-01 00:18:51,2024-03-01 00:23:45,142,239,1.30,8.6,0.0
1,2024-03-01 00:26:00,2024-03-01 00:29:06,238,24,1.10,7.2,0.0
2,2024-03-01 00:09:22,2024-03-01 00:15:24,263,75,0.86,7.9,1.0
3,2024-03-01 00:33:45,2024-03-01 00:39:34,164,162,0.82,7.9,1.0
4,2024-03-01 00:05:43,2024-03-01 00:26:22,263,7,4.90,25.4,0.0


In [84]:
df.to_sql(
    "nyc_taxi_raw",
    engine,
    if_exists="replace",
    index=False,
    chunksize=10_000,
)

print("NYC Taxi data loaded successfully!")


NYC Taxi data loaded successfully!


In [50]:
%pip install --force-reinstall "prettytable==3.11.0"


  Using cached prettytable-3.11.0-py3-none-any.whl.metadata (30 kB)
  Using cached wcwidth-0.9.4-cp310-abi3-macosx_11_0_arm64.whl.metadata (23 kB)
Using cached prettytable-3.11.0-py3-none-any.whl (28 kB)
Using cached wcwidth-0.9.4-cp310-abi3-macosx_11_0_arm64.whl (618 kB)
  Attempting uninstall: wcwidth
    Found existing installation: wcwidth 0.9.4
    Uninstalling wcwidth-0.9.4:
      Successfully uninstalled wcwidth-0.9.4
  Attempting uninstall: prettytable━━━━━━━━━━━━━ 0/2 [wcwidth]
    Found existing installation: prettytable 3.11.032m0/2 [wcwidth]
    Uninstalling prettytable-3.11.0:━━━━━━━━ 0/2 [wcwidth]
      Successfully uninstalled prettytable-3.11.00/2 [wcwidth]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [prettytable]
Note: you may need to restart the kernel to use updated packages.


In [52]:
%load_ext sql

db_url = pg.get_uri()
%sql $db_url


The sql extension is already loaded. To reload it, use:
  %reload_ext sql


In [53]:
%config SqlMagic.style = 'DEFAULT'


Дані за лютий 2024 року

In [100]:
from pathlib import Path
from urllib.request import urlretrieve

data_dir = Path("data")
data_dir.mkdir(exist_ok=True)

feb_path = data_dir / "yellow_tripdata_2024-02.parquet"

URL_FEB = (
    "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-02.parquet"
)

if not feb_path.exists():
    urlretrieve(URL_FEB, feb_path)

print("February file:", feb_path)
print("File exists:", feb_path.exists())


February file: data/yellow_tripdata_2024-02.parquet
File exists: True


In [113]:
import pandas as pd

march_path = Path("data/yellow_tripdata_2024-03.parquet")

daily_columns = [
    "tpep_pickup_datetime",
    "PULocationID",
    "trip_distance",
    "fare_amount",
]


def prepare_daily_data(path):
    df = pd.read_parquet(path, columns=daily_columns)

    # Відбираємо лише лютий і березень 2024 року
    df = df[
        (df["tpep_pickup_datetime"] >= "2024-02-01")
        & (df["tpep_pickup_datetime"] < "2024-04-01")
    ].copy()

    df = df[
        (df["fare_amount"] > 0)
        & (df["fare_amount"] < 500)
        & (df["trip_distance"] > 0)
        & (df["trip_distance"] < 100)
        & (df["PULocationID"].notna())
    ].copy()

    df["ts_day"] = df["tpep_pickup_datetime"].dt.date

    daily = (
        df.groupby(["PULocationID", "ts_day"])
        .agg(
            trip_count=("fare_amount", "size"),
            avg_fare=("fare_amount", "mean"),
            total_distance=("trip_distance", "sum"),
        )
        .reset_index()
        .rename(columns={"PULocationID": "pickup_zone_id"})
    )

    return daily


feb_daily = prepare_daily_data(feb_path)
march_daily = prepare_daily_data(march_path)

daily_full = pd.concat([feb_daily, march_daily], ignore_index=True)

# Повторна агрегація для однакових зон і дат
daily_full["total_fare"] = daily_full["avg_fare"] * daily_full["trip_count"]

daily_full = daily_full.groupby(["pickup_zone_id", "ts_day"], as_index=False).agg(
    trip_count=("trip_count", "sum"),
    total_fare=("total_fare", "sum"),
    total_distance=("total_distance", "sum"),
)

daily_full["avg_fare"] = daily_full["total_fare"] / daily_full["trip_count"]

daily_full = daily_full.drop(columns=["total_fare"])

print("Rows:", len(daily_full))
print("Duplicates:", daily_full.duplicated(["pickup_zone_id", "ts_day"]).sum())

display(daily_full.head())


Rows: 13511
Duplicates: 0


,pickup_zone_id,ts_day,trip_count,total_distance,avg_fare
0,1,2024-02-01,2,17.83,117.5
1,1,2024-02-04,3,17.38,45.1
2,1,2024-02-05,1,2.50,125.0
3,1,2024-02-07,1,0.12,110.0
4,1,2024-02-08,3,7.05,92.4


In [114]:
%%sql

WITH lag_features AS (
    SELECT
        pickup_zone_id,
        ts_day,
        trip_count,
        LAG(trip_count, 1) OVER w AS trips_lag_1d,
        LAG(trip_count, 7) OVER w AS trips_lag_7d,
        LAG(trip_count, 30) OVER w AS trips_lag_30d
    FROM t7_hw_taxi_daily
    WINDOW w AS (
        PARTITION BY pickup_zone_id
        ORDER BY ts_day
    )
)
SELECT *
FROM lag_features
WHERE trips_lag_30d IS NOT NULL
ORDER BY pickup_zone_id, ts_day
LIMIT 20;

 * postgresql://postgres:***@/postgres?host=%2Fprivate%2Ftmp%2Fhw7_pg
20 rows affected.


pickup_zone_id,ts_day,trip_count,trips_lag_1d,trips_lag_7d,trips_lag_30d
1,2024-03-11,3,3,1,2
1,2024-03-12,1,3,2,3
1,2024-03-13,1,1,2,1
1,2024-03-14,5,1,1,1
1,2024-03-15,6,5,1,3
1,2024-03-16,4,6,2,1
1,2024-03-17,4,4,3,3
1,2024-03-18,5,4,3,2
1,2024-03-19,1,5,1,1
1,2024-03-20,3,1,1,2


In [115]:
from sqlalchemy import Integer, Date, Numeric

daily_full.to_sql(
    "t7_hw_taxi_daily",
    engine,
    if_exists="replace",
    index=False,
    chunksize=10_000,
    dtype={
        "pickup_zone_id": Integer(),
        "ts_day": Date(),
        "trip_count": Integer(),
        "avg_fare": Numeric(10, 2),
        "total_distance": Numeric(12, 2),
    },
)

print("Daily table updated successfully!")


Daily table updated successfully!


In [116]:
%%sql

ALTER TABLE t7_hw_taxi_daily
ADD PRIMARY KEY (pickup_zone_id, ts_day);

 * postgresql://postgres:***@/postgres?host=%2Fprivate%2Ftmp%2Fhw7_pg
Done.


[]

In [117]:
%%sql

DROP TABLE IF EXISTS t7_hw_taxi_trips_typed;

CREATE TABLE t7_hw_taxi_trips_typed AS
SELECT
    "PULocationID"::INTEGER AS pickup_zone_id,
    "DOLocationID"::INTEGER AS dropoff_zone_id,

    (
        tpep_pickup_datetime::TIMESTAMP
        AT TIME ZONE 'America/New_York'
    ) AS pickup_at,

    (
        tpep_dropoff_datetime::TIMESTAMP
        AT TIME ZONE 'America/New_York'
    ) AS dropoff_at,

    trip_distance::NUMERIC(10, 2) AS trip_distance,
    fare_amount::NUMERIC(10, 2) AS fare_amount,
    passenger_count::INTEGER AS passenger_count

FROM nyc_taxi_raw

WHERE fare_amount > 0
  AND fare_amount < 500
  AND trip_distance > 0
  AND trip_distance < 100;

 * postgresql://postgres:***@/postgres?host=%2Fprivate%2Ftmp%2Fhw7_pg
Done.
486964 rows affected.


[]

Завдання 2. Time-series ML features



Feature 1. Lag features: lag-1d, lag-7d, lag-30d



In [119]:
%%sql

WITH lag_features AS (
    SELECT
        pickup_zone_id,
        ts_day,
        trip_count,

        LAG(trip_count, 1) OVER w AS trips_lag_1d,
        LAG(trip_count, 7) OVER w AS trips_lag_7d,
        LAG(trip_count, 30) OVER w AS trips_lag_30d

    FROM t7_hw_taxi_daily

    WINDOW w AS (
        PARTITION BY pickup_zone_id
        ORDER BY ts_day
    )
)

SELECT *
FROM lag_features
WHERE trips_lag_30d IS NOT NULL
ORDER BY pickup_zone_id, ts_day
LIMIT 20;

 * postgresql://postgres:***@/postgres?host=%2Fprivate%2Ftmp%2Fhw7_pg
20 rows affected.


pickup_zone_id,ts_day,trip_count,trips_lag_1d,trips_lag_7d,trips_lag_30d
1,2024-03-11,3,3,1,2
1,2024-03-12,1,3,2,3
1,2024-03-13,1,1,2,1
1,2024-03-14,5,1,1,1
1,2024-03-15,6,5,1,3
1,2024-03-16,4,6,2,1
1,2024-03-17,4,4,3,3
1,2024-03-18,5,4,3,2
1,2024-03-19,1,5,1,1
1,2024-03-20,3,1,1,2


Feature 2. Rolling 7-day average fare



In [120]:
%%sql

SELECT
    pickup_zone_id,
    ts_day,
    avg_fare,

    AVG(avg_fare) OVER w7 AS rolling_avg_fare_7d,
    COUNT(*) OVER w7 AS days_in_window

FROM t7_hw_taxi_daily

WINDOW w7 AS (
    PARTITION BY pickup_zone_id
    ORDER BY ts_day::TIMESTAMP
    RANGE BETWEEN INTERVAL '6 days' PRECEDING
              AND CURRENT ROW
)

ORDER BY pickup_zone_id, ts_day
LIMIT 20;

 * postgresql://postgres:***@/postgres?host=%2Fprivate%2Ftmp%2Fhw7_pg
20 rows affected.


pickup_zone_id,ts_day,avg_fare,rolling_avg_fare_7d,days_in_window
1,2024-02-01,117.50,117.5000000000000000,1
1,2024-02-04,45.10,81.3000000000000000,2
1,2024-02-05,125.00,95.8666666666666667,3
1,2024-02-07,110.00,99.4000000000000000,4
1,2024-02-08,92.40,93.1250000000000000,4
1,2024-02-09,10.00,76.5000000000000000,5
1,2024-02-11,26.67,72.8140000000000000,5
1,2024-02-12,97.50,67.3140000000000000,5
1,2024-02-14,105.50,66.4140000000000000,5
1,2024-02-15,42.20,56.3740000000000000,5


Повні 7-денні вікна

In [121]:
%%sql

WITH rolling_features AS (
    SELECT
        pickup_zone_id,
        ts_day,
        avg_fare,
        AVG(avg_fare) OVER w7 AS rolling_avg_fare_7d,
        COUNT(*) OVER w7 AS days_in_window
    FROM t7_hw_taxi_daily
    WINDOW w7 AS (
        PARTITION BY pickup_zone_id
        ORDER BY ts_day::TIMESTAMP
        RANGE BETWEEN INTERVAL '6 days' PRECEDING
                  AND CURRENT ROW
    )
)

SELECT *
FROM rolling_features
WHERE days_in_window = 7
ORDER BY pickup_zone_id, ts_day
LIMIT 20;

 * postgresql://postgres:***@/postgres?host=%2Fprivate%2Ftmp%2Fhw7_pg
20 rows affected.


pickup_zone_id,ts_day,avg_fare,rolling_avg_fare_7d,days_in_window
1,2024-02-20,77.43,69.9457142857142857,7
1,2024-02-21,86.43,67.2214285714285714,7
1,2024-02-22,80.00,72.6214285714285714,7
1,2024-02-23,117.45,78.6857142857142857,7
1,2024-02-24,96.85,83.7814285714285714,7
1,2024-02-25,50.60,78.8985714285714286,7
1,2024-02-26,170.00,96.9657142857142857,7
1,2024-02-27,118.75,102.8685714285714286,7
1,2024-02-28,114.16,106.8300000000000000,7
1,2024-03-16,123.68,83.5700000000000000,7


Feature 3. Rolling 30-day fare volatility

In [122]:
%%sql

SELECT
    pickup_zone_id,
    ts_day,
    avg_fare,

    STDDEV_SAMP(avg_fare) OVER w30 AS rolling_std_fare_30d,
    COUNT(*) OVER w30 AS days_in_window

FROM t7_hw_taxi_daily

WINDOW w30 AS (
    PARTITION BY pickup_zone_id
    ORDER BY ts_day::TIMESTAMP
    RANGE BETWEEN INTERVAL '29 days' PRECEDING
              AND CURRENT ROW
)

ORDER BY pickup_zone_id, ts_day
LIMIT 20;

 * postgresql://postgres:***@/postgres?host=%2Fprivate%2Ftmp%2Fhw7_pg
20 rows affected.


pickup_zone_id,ts_day,avg_fare,rolling_std_fare_30d,days_in_window
1,2024-02-01,117.50,None,1
1,2024-02-04,45.10,51.1945309579060408,2
1,2024-02-05,125.00,44.1248607174383422,3
1,2024-02-07,110.00,36.7143023902129999,4
1,2024-02-08,92.40,31.9492566423696312,5
1,2024-02-09,10.00,45.9050178811278394,6
1,2024-02-11,26.67,47.0609719507829646,7
1,2024-02-12,97.50,44.2751708676222147,8
1,2024-02-14,105.50,42.4164163712327043,9
1,2024-02-15,42.20,41.8374091108795970,10


Feature 4. Expanding cumulative trips

In [123]:
%%sql

SELECT
    pickup_zone_id,
    ts_day,
    trip_count,

    SUM(trip_count) OVER we AS cumulative_trips,
    AVG(trip_count) OVER we AS expanding_avg_trips

FROM t7_hw_taxi_daily

WINDOW we AS (
    PARTITION BY pickup_zone_id
    ORDER BY ts_day
    ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
)

ORDER BY pickup_zone_id, ts_day
LIMIT 20;

 * postgresql://postgres:***@/postgres?host=%2Fprivate%2Ftmp%2Fhw7_pg
20 rows affected.


pickup_zone_id,ts_day,trip_count,cumulative_trips,expanding_avg_trips
1,2024-02-01,2,2,2.0000000000000000
1,2024-02-04,3,5,2.5000000000000000
1,2024-02-05,1,6,2.0000000000000000
1,2024-02-07,1,7,1.7500000000000000
1,2024-02-08,3,10,2.0000000000000000
1,2024-02-09,1,11,1.8333333333333333
1,2024-02-11,3,14,2.0000000000000000
1,2024-02-12,2,16,2.0000000000000000
1,2024-02-14,1,17,1.8888888888888889
1,2024-02-15,2,19,1.9000000000000000


Combined feature view



In [124]:
%%sql

CREATE OR REPLACE VIEW v_t7_hw_taxi_features AS

WITH features_basic AS (
    SELECT
        pickup_zone_id,
        ts_day,
        trip_count,
        avg_fare,
        total_distance,

        -- Feature 1: Lag features
        LAG(trip_count, 1) OVER w_pid AS trips_lag_1d,
        LAG(trip_count, 7) OVER w_pid AS trips_lag_7d,
        LAG(trip_count, 30) OVER w_pid AS trips_lag_30d,

        -- Feature 2: Rolling 7-day average
        AVG(avg_fare) OVER w7 AS rolling_avg_fare_7d,

        -- Feature 3: Rolling 30-day volatility
        STDDEV_SAMP(avg_fare) OVER w30 AS rolling_std_fare_30d,

        -- Feature 4: Expanding features
        SUM(trip_count) OVER we AS cumulative_trips,
        AVG(trip_count) OVER we AS expanding_avg_trips

    FROM t7_hw_taxi_daily

    WINDOW
        w_pid AS (
            PARTITION BY pickup_zone_id
            ORDER BY ts_day
        ),

        w7 AS (
            PARTITION BY pickup_zone_id
            ORDER BY ts_day::TIMESTAMP
            RANGE BETWEEN INTERVAL '6 days' PRECEDING
                      AND CURRENT ROW
        ),

        w30 AS (
            PARTITION BY pickup_zone_id
            ORDER BY ts_day::TIMESTAMP
            RANGE BETWEEN INTERVAL '29 days' PRECEDING
                      AND CURRENT ROW
        ),

        we AS (
            PARTITION BY pickup_zone_id
            ORDER BY ts_day
            ROWS BETWEEN UNBOUNDED PRECEDING
                     AND CURRENT ROW
        )
)

SELECT *
FROM features_basic
WHERE trips_lag_30d IS NOT NULL;

 * postgresql://postgres:***@/postgres?host=%2Fprivate%2Ftmp%2Fhw7_pg
Done.


[]

In [125]:
%%sql

SELECT *
FROM v_t7_hw_taxi_features
ORDER BY pickup_zone_id, ts_day
LIMIT 20;

 * postgresql://postgres:***@/postgres?host=%2Fprivate%2Ftmp%2Fhw7_pg
20 rows affected.


pickup_zone_id,ts_day,trip_count,avg_fare,total_distance,trips_lag_1d,trips_lag_7d,trips_lag_30d,rolling_avg_fare_7d,rolling_std_fare_30d,cumulative_trips,expanding_avg_trips
1,2024-03-11,3,44.33,19.61,3,1,2,76.8025000000000000,33.4165429939523617,72,2.3225806451612903
1,2024-03-12,1,117.00,0.72,3,2,3,76.0525000000000000,32.0574681262676555,73,2.2812500000000000
1,2024-03-13,1,117.00,0.09,1,2,1,84.2420000000000000,32.5952713237160624,74,2.2424242424242424
1,2024-03-14,5,51.20,20.45,1,1,1,78.7350000000000000,32.6677873447223236,79,2.3235294117647059
1,2024-03-15,6,71.45,87.20,5,1,3,76.8850000000000000,32.4907481269076635,85,2.4285714285714286
1,2024-03-16,4,123.68,53.22,6,2,1,83.5700000000000000,32.2680648224310995,89,2.4722222222222222
1,2024-03-17,4,115.40,35.60,4,3,3,91.4371428571428571,32.6515977886817379,93,2.5135135135135135
1,2024-03-18,5,98.11,52.01,4,3,2,99.1200000000000000,32.2263534322216099,98,2.5789473684210526
1,2024-03-19,1,101.00,18.01,5,1,1,96.8342857142857143,32.2843759806742813,99,2.5384615384615385
1,2024-03-20,3,76.17,20.65,1,1,2,91.0014285714285714,31.0013817185869711,102,2.5500000000000000


Завдання 3. JSONB-таблиця для подій



In [126]:
%%sql

DROP TABLE IF EXISTS t7_hw_events;

CREATE TABLE t7_hw_events (
    event_id INTEGER GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
    user_id INTEGER NOT NULL,
    event_ts TIMESTAMPTZ NOT NULL DEFAULT NOW(),
    event_type TEXT NOT NULL,
    payload JSONB NOT NULL
);

 * postgresql://postgres:***@/postgres?host=%2Fprivate%2Ftmp%2Fhw7_pg
Done.
Done.


[]

In [127]:
%%sql
INSERT INTO t7_hw_events (user_id, event_ts, event_type, payload)
SELECT
    (n % 1000) + 1 AS user_id,
    TIMESTAMPTZ '2024-04-01 00:00:00+00'
        - ((n % 60) || ' days')::INTERVAL AS event_ts,
    CASE (n % 5)
        WHEN 0 THEN 'page_view'
        WHEN 1 THEN 'click'
        WHEN 2 THEN 'add_to_cart'
        WHEN 3 THEN 'checkout'
        ELSE        'recommendation_shown'
    END AS event_type,
    CASE (n % 5)
        WHEN 0 THEN jsonb_build_object(
            'url', '/product/' || ((n % 200) + 1),
            'referrer', (ARRAY['google','direct','email','fb'])[(n % 4) + 1],
            'device', (ARRAY['mobile','desktop','tablet'])[(n % 3) + 1]
        )
        WHEN 1 THEN jsonb_build_object(
            'element_id', 'btn_' || ((n % 50) + 1),
            'page_url', '/category/' || ((n % 20) + 1),
            'device', (ARRAY['mobile','desktop'])[(n % 2) + 1]
        )
        WHEN 2 THEN jsonb_build_object(
            'product_id', (n % 200) + 1,
            'price', ((n % 100) + 10)::NUMERIC(10, 2),
            'quantity', (n % 5) + 1,
            'tags', jsonb_build_array(
                (ARRAY['promo','new','clearance'])[(n % 3) + 1]
            )
        )
        WHEN 3 THEN jsonb_build_object(
            'order_id', 'o_' || (n + 100000),
            'total', ((n % 500) + 50)::NUMERIC(10, 2),
            'items_count', (n % 10) + 1,
            'payment_method', (ARRAY['card','paypal','crypto'])[(n % 3) + 1]
        )
        ELSE jsonb_build_object(
            'model_name', 'rec_v' || ((n % 5) + 1),
            'model_version', '1.' || (n % 10),
            'recommendations', jsonb_build_array(
                (n % 200) + 1,
                ((n + 1) % 200) + 1,
                ((n + 2) % 200) + 1
            ),
            'experiment', 'exp_' || ((n % 3) + 1)
        )
    END AS payload
FROM generate_series(1, 50000) AS n;


 * postgresql://postgres:***@/postgres?host=%2Fprivate%2Ftmp%2Fhw7_pg
50000 rows affected.


[]

In [128]:
%%sql

ANALYZE t7_hw_events;

SELECT
    event_type,
    COUNT(*) AS n_events
FROM t7_hw_events
GROUP BY event_type
ORDER BY event_type;

 * postgresql://postgres:***@/postgres?host=%2Fprivate%2Ftmp%2Fhw7_pg
Done.
5 rows affected.


event_type,n_events
add_to_cart,10000
checkout,10000
click,10000
page_view,10000
recommendation_shown,10000


In [129]:
%%sql

SELECT DISTINCT ON (event_type)
    event_type,
    payload,
    pg_typeof(payload) AS payload_type
FROM t7_hw_events
ORDER BY event_type, event_id;

 * postgresql://postgres:***@/postgres?host=%2Fprivate%2Ftmp%2Fhw7_pg
5 rows affected.


event_type,payload,payload_type
add_to_cart,"{'tags': ['clearance'], 'price': 12.0, 'quantity': 3, 'product_id': 3}",jsonb
checkout,"{'total': 53.0, 'order_id': 'o_100003', 'items_count': 4, 'payment_method': 'card'}",jsonb
click,"{'device': 'desktop', 'page_url': '/category/2', 'element_id': 'btn_2'}",jsonb
page_view,"{'url': '/product/6', 'device': 'tablet', 'referrer': 'direct'}",jsonb
recommendation_shown,"{'experiment': 'exp_2', 'model_name': 'rec_v5', 'model_version': '1.4', 'recommendations': [5, 6, 7]}",jsonb


JSONB Query 1. Containment

In [130]:
%%sql

SELECT
    event_id,
    user_id,
    event_ts,
    payload
FROM t7_hw_events
WHERE event_type = 'checkout'
  AND payload @> jsonb_build_object('payment_method', 'card')
LIMIT 10;

 * postgresql://postgres:***@/postgres?host=%2Fprivate%2Ftmp%2Fhw7_pg
10 rows affected.


event_id,user_id,event_ts,payload
3,4,2024-03-29 02:00:00+01:00,"{'total': 53.0, 'order_id': 'o_100003', 'items_count': 4, 'payment_method': 'card'}"
18,19,2024-03-14 02:00:00+01:00,"{'total': 68.0, 'order_id': 'o_100018', 'items_count': 9, 'payment_method': 'card'}"
33,34,2024-02-28 02:00:00+01:00,"{'total': 83.0, 'order_id': 'o_100033', 'items_count': 4, 'payment_method': 'card'}"
48,49,2024-02-13 02:00:00+01:00,"{'total': 98.0, 'order_id': 'o_100048', 'items_count': 9, 'payment_method': 'card'}"
63,64,2024-03-29 02:00:00+01:00,"{'total': 113.0, 'order_id': 'o_100063', 'items_count': 4, 'payment_method': 'card'}"
78,79,2024-03-14 02:00:00+01:00,"{'total': 128.0, 'order_id': 'o_100078', 'items_count': 9, 'payment_method': 'card'}"
93,94,2024-02-28 02:00:00+01:00,"{'total': 143.0, 'order_id': 'o_100093', 'items_count': 4, 'payment_method': 'card'}"
108,109,2024-02-13 02:00:00+01:00,"{'total': 158.0, 'order_id': 'o_100108', 'items_count': 9, 'payment_method': 'card'}"
123,124,2024-03-29 02:00:00+01:00,"{'total': 173.0, 'order_id': 'o_100123', 'items_count': 4, 'payment_method': 'card'}"
138,139,2024-03-14 02:00:00+01:00,"{'total': 188.0, 'order_id': 'o_100138', 'items_count': 9, 'payment_method': 'card'}"


JSONB Query 2. Key existence ? та ?|



In [131]:
%%sql

SELECT
    event_id,
    event_type,
    payload
FROM t7_hw_events
WHERE payload ? 'tags'
LIMIT 10;

 * postgresql://postgres:***@/postgres?host=%2Fprivate%2Ftmp%2Fhw7_pg
10 rows affected.


event_id,event_type,payload
2,add_to_cart,"{'tags': ['clearance'], 'price': 12.0, 'quantity': 3, 'product_id': 3}"
7,add_to_cart,"{'tags': ['new'], 'price': 17.0, 'quantity': 3, 'product_id': 8}"
12,add_to_cart,"{'tags': ['promo'], 'price': 22.0, 'quantity': 3, 'product_id': 13}"
17,add_to_cart,"{'tags': ['clearance'], 'price': 27.0, 'quantity': 3, 'product_id': 18}"
22,add_to_cart,"{'tags': ['new'], 'price': 32.0, 'quantity': 3, 'product_id': 23}"
27,add_to_cart,"{'tags': ['promo'], 'price': 37.0, 'quantity': 3, 'product_id': 28}"
32,add_to_cart,"{'tags': ['clearance'], 'price': 42.0, 'quantity': 3, 'product_id': 33}"
37,add_to_cart,"{'tags': ['new'], 'price': 47.0, 'quantity': 3, 'product_id': 38}"
42,add_to_cart,"{'tags': ['promo'], 'price': 52.0, 'quantity': 3, 'product_id': 43}"
47,add_to_cart,"{'tags': ['clearance'], 'price': 57.0, 'quantity': 3, 'product_id': 48}"


In [132]:
%%sql

SELECT
    event_id,
    event_type,
    payload
FROM t7_hw_events
WHERE payload ?| ARRAY['experiment', 'model_version']
LIMIT 10;

 * postgresql://postgres:***@/postgres?host=%2Fprivate%2Ftmp%2Fhw7_pg
10 rows affected.


event_id,event_type,payload
4,recommendation_shown,"{'experiment': 'exp_2', 'model_name': 'rec_v5', 'model_version': '1.4', 'recommendations': [5, 6, 7]}"
9,recommendation_shown,"{'experiment': 'exp_1', 'model_name': 'rec_v5', 'model_version': '1.9', 'recommendations': [10, 11, 12]}"
14,recommendation_shown,"{'experiment': 'exp_3', 'model_name': 'rec_v5', 'model_version': '1.4', 'recommendations': [15, 16, 17]}"
19,recommendation_shown,"{'experiment': 'exp_2', 'model_name': 'rec_v5', 'model_version': '1.9', 'recommendations': [20, 21, 22]}"
24,recommendation_shown,"{'experiment': 'exp_1', 'model_name': 'rec_v5', 'model_version': '1.4', 'recommendations': [25, 26, 27]}"
29,recommendation_shown,"{'experiment': 'exp_3', 'model_name': 'rec_v5', 'model_version': '1.9', 'recommendations': [30, 31, 32]}"
34,recommendation_shown,"{'experiment': 'exp_2', 'model_name': 'rec_v5', 'model_version': '1.4', 'recommendations': [35, 36, 37]}"
39,recommendation_shown,"{'experiment': 'exp_1', 'model_name': 'rec_v5', 'model_version': '1.9', 'recommendations': [40, 41, 42]}"
44,recommendation_shown,"{'experiment': 'exp_3', 'model_name': 'rec_v5', 'model_version': '1.4', 'recommendations': [45, 46, 47]}"
49,recommendation_shown,"{'experiment': 'exp_2', 'model_name': 'rec_v5', 'model_version': '1.9', 'recommendations': [50, 51, 52]}"


In [133]:
%%sql

SELECT
    event_id,
    event_type,
    payload
FROM t7_hw_events
WHERE payload ?& ARRAY['experiment', 'model_version']
LIMIT 10;

 * postgresql://postgres:***@/postgres?host=%2Fprivate%2Ftmp%2Fhw7_pg
10 rows affected.


event_id,event_type,payload
4,recommendation_shown,"{'experiment': 'exp_2', 'model_name': 'rec_v5', 'model_version': '1.4', 'recommendations': [5, 6, 7]}"
9,recommendation_shown,"{'experiment': 'exp_1', 'model_name': 'rec_v5', 'model_version': '1.9', 'recommendations': [10, 11, 12]}"
14,recommendation_shown,"{'experiment': 'exp_3', 'model_name': 'rec_v5', 'model_version': '1.4', 'recommendations': [15, 16, 17]}"
19,recommendation_shown,"{'experiment': 'exp_2', 'model_name': 'rec_v5', 'model_version': '1.9', 'recommendations': [20, 21, 22]}"
24,recommendation_shown,"{'experiment': 'exp_1', 'model_name': 'rec_v5', 'model_version': '1.4', 'recommendations': [25, 26, 27]}"
29,recommendation_shown,"{'experiment': 'exp_3', 'model_name': 'rec_v5', 'model_version': '1.9', 'recommendations': [30, 31, 32]}"
34,recommendation_shown,"{'experiment': 'exp_2', 'model_name': 'rec_v5', 'model_version': '1.4', 'recommendations': [35, 36, 37]}"
39,recommendation_shown,"{'experiment': 'exp_1', 'model_name': 'rec_v5', 'model_version': '1.9', 'recommendations': [40, 41, 42]}"
44,recommendation_shown,"{'experiment': 'exp_3', 'model_name': 'rec_v5', 'model_version': '1.4', 'recommendations': [45, 46, 47]}"
49,recommendation_shown,"{'experiment': 'exp_2', 'model_name': 'rec_v5', 'model_version': '1.9', 'recommendations': [50, 51, 52]}"


JSONB Query 3. JSONPath



1. Extracting elements of a JSON array

In [ ]:
%%sql

SELECT
    event_id,
    jsonb_path_query(
        payload,
        '$.recommendations[*]'
    ) AS recommended_pid
FROM t7_hw_events
WHERE event_type = 'recommendation_shown'
LIMIT 15;

2. Filtering JSON based on a condition

In [134]:
%%sql

SELECT
    event_id,
    payload ->> 'order_id' AS order_id,
    (payload ->> 'total')::NUMERIC(10, 2) AS total
FROM t7_hw_events
WHERE event_type = 'checkout'
  AND jsonb_path_exists(
      payload,
      '$.total ? (@ > 300)'
  )
LIMIT 10;

 * postgresql://postgres:***@/postgres?host=%2Fprivate%2Ftmp%2Fhw7_pg
10 rows affected.


event_id,order_id,total
253,o_100253,303.00
258,o_100258,308.00
263,o_100263,313.00
268,o_100268,318.00
273,o_100273,323.00
278,o_100278,328.00
283,o_100283,333.00
288,o_100288,338.00
293,o_100293,343.00
298,o_100298,348.00


Task 4. GIN-index demo + EXPLAIN ANALYZE

Step 1. Containment query without GIN index


In [135]:
%%sql

DROP INDEX IF EXISTS idx_t7_events_payload_gin_pathops;
DROP INDEX IF EXISTS idx_t7_events_payload_gin_ops;
DROP INDEX IF EXISTS idx_t7_events_device_btree;

ANALYZE t7_hw_events;

EXPLAIN (ANALYZE, BUFFERS)
SELECT COUNT(*)
FROM t7_hw_events
WHERE payload @> jsonb_build_object('payment_method', 'card');

 * postgresql://postgres:***@/postgres?host=%2Fprivate%2Ftmp%2Fhw7_pg
Done.
Done.
Done.
Done.
10 rows affected.


QUERY PLAN
Aggregate (cost=1759.04..1759.05 rows=1 width=8) (actual time=18.957..19.062 rows=1 loops=1)
Buffers: shared hit=1000
-> Seq Scan on t7_hw_events (cost=0.00..1750.00 rows=3616 width=0) (actual time=0.005..18.837 rows=3334 loops=1)
"Filter: (payload @> jsonb_build_object('payment_method', 'card'))"
Rows Removed by Filter: 46666
Buffers: shared hit=1000
Planning:
Buffers: shared hit=5
Planning Time: 2.560 ms
Execution Time: 19.788 ms


**Крок 1. Запит на перевірку вмісту (containment query) без індексу GIN**

Перед створенням індексу GIN я скористався командою `EXPLAIN (ANALYZE, BUFFERS)` для аналізу запиту на перевірку вмісту (containment query) для типу JSONB.

Очікуваний план виконання — послідовне сканування (`Seq Scan`), під час якого PostgreSQL переглядає таблицю та для кожного рядка перевіряє умову `payment_method = 'card'`.

Цей вимір слугує базовим показником для порівняння продуктивності запиту до та після створення індексу. Ключовими метриками є час виконання, тип сканування та використання буферів.

На наступному етапі отримані результати будуть порівняні з результатами виконання запиту за наявності індексу.

Step 2. GIN index for containment / JSONPath workload

In [136]:
%%sql

CREATE INDEX idx_t7_events_payload_gin_pathops
    ON t7_hw_events
    USING GIN (payload jsonb_path_ops);

ANALYZE t7_hw_events;

SELECT
    pg_size_pretty(
        pg_relation_size('idx_t7_events_payload_gin_pathops')
    ) AS gin_index_size;

 * postgresql://postgres:***@/postgres?host=%2Fprivate%2Ftmp%2Fhw7_pg
Done.
Done.
1 rows affected.


gin_index_size
1144 kB


Step 3. The same containment query after GIN

In [137]:
%%sql

EXPLAIN (ANALYZE, BUFFERS)
SELECT COUNT(*)
FROM t7_hw_events
WHERE payload @> jsonb_build_object('payment_method', 'card');

 * postgresql://postgres:***@/postgres?host=%2Fprivate%2Ftmp%2Fhw7_pg
13 rows affected.


QUERY PLAN
Aggregate (cost=1099.37..1099.38 rows=1 width=8) (actual time=17.145..17.148 rows=1 loops=1)
Buffers: shared hit=1004
-> Bitmap Heap Scan on t7_hw_events (cost=35.97..1090.32 rows=3623 width=0) (actual time=2.673..16.971 rows=3334 loops=1)
"Recheck Cond: (payload @> jsonb_build_object('payment_method', 'card'))"
Heap Blocks: exact=1000
Buffers: shared hit=1004
-> Bitmap Index Scan on idx_t7_events_payload_gin_pathops (cost=0.00..35.07 rows=3623 width=0) (actual time=2.446..2.446 rows=3334 loops=1)
"Index Cond: (payload @> jsonb_build_object('payment_method', 'card'))"
Buffers: shared hit=4
Planning:


In [138]:
%%sql

EXPLAIN (ANALYZE, BUFFERS)
SELECT COUNT(*)
FROM t7_hw_events
WHERE payload @> jsonb_build_object('referrer', 'google');

 * postgresql://postgres:***@/postgres?host=%2Fprivate%2Ftmp%2Fhw7_pg
13 rows affected.


QUERY PLAN
Aggregate (cost=1067.47..1067.48 rows=1 width=8) (actual time=5.189..5.190 rows=1 loops=1)
Buffers: shared hit=1004
-> Bitmap Heap Scan on t7_hw_events (cost=25.44..1061.47 rows=2402 width=0) (actual time=0.514..5.064 rows=2500 loops=1)
"Recheck Cond: (payload @> jsonb_build_object('referrer', 'google'))"
Heap Blocks: exact=1000
Buffers: shared hit=1004
-> Bitmap Index Scan on idx_t7_events_payload_gin_pathops (cost=0.00..24.83 rows=2402 width=0) (actual time=0.354..0.354 rows=2500 loops=1)
"Index Cond: (payload @> jsonb_build_object('referrer', 'google'))"
Buffers: shared hit=4
Planning:


Step 4. When GIN doesn't help: path-specific text equality

In [139]:
%%sql

EXPLAIN (ANALYZE, BUFFERS)
SELECT COUNT(*)
FROM t7_hw_events
WHERE payload ->> 'device' = 'mobile';

 * postgresql://postgres:***@/postgres?host=%2Fprivate%2Ftmp%2Fhw7_pg
8 rows affected.


QUERY PLAN
Aggregate (cost=1750.62..1750.63 rows=1 width=8) (actual time=10.847..10.849 rows=1 loops=1)
Buffers: shared hit=1000
-> Seq Scan on t7_hw_events (cost=0.00..1750.00 rows=250 width=0) (actual time=0.052..10.541 rows=8333 loops=1)
Filter: ((payload ->> 'device'::text) = 'mobile'::text)
Rows Removed by Filter: 41667
Buffers: shared hit=1000
Planning Time: 0.561 ms
Execution Time: 11.097 ms


expression B-tree index

In [140]:
%%sql

CREATE INDEX idx_t7_events_device_btree
    ON t7_hw_events ((payload ->> 'device'));

ANALYZE t7_hw_events;

 * postgresql://postgres:***@/postgres?host=%2Fprivate%2Ftmp%2Fhw7_pg
Done.
Done.


[]

Let's revisit EXPLAIN ANALYZE

In [141]:
%%sql

EXPLAIN (ANALYZE, BUFFERS)
SELECT COUNT(*)
FROM t7_hw_events
WHERE payload ->> 'device' = 'mobile';

 * postgresql://postgres:***@/postgres?host=%2Fprivate%2Ftmp%2Fhw7_pg
13 rows affected.


QUERY PLAN
Aggregate (cost=1242.75..1242.76 rows=1 width=8) (actual time=5.521..5.522 rows=1 loops=1)
Buffers: shared hit=1 read=9
-> Bitmap Heap Scan on t7_hw_events (cost=96.89..1221.91 rows=8335 width=0) (actual time=4.818..5.229 rows=8333 loops=1)
Recheck Cond: ((payload ->> 'device'::text) = 'mobile'::text)
Heap Blocks: exact=1000
Buffers: shared hit=1 read=9
-> Bitmap Index Scan on idx_t7_events_device_btree (cost=0.00..94.80 rows=8335 width=0) (actual time=2.980..2.980 rows=8333 loops=1)
Index Cond: ((payload ->> 'device'::text) = 'mobile'::text)
Buffers: shared read=9
Planning:


## Task 4. Reflection — GIN Index Demo + EXPLAIN ANALYZE

У цьому завданні я дослідила вплив GIN та B-tree індексів на продуктивність JSONB-запитів у PostgreSQL, використовуючи `EXPLAIN (ANALYZE, BUFFERS)`.

### Step 1–3. Порівняння JSONB containment query до та після GIN

Спочатку я виконала запит із фільтром `payment_method = 'card'` без GIN-індексу. PostgreSQL використав **Seq Scan** з оціненою вартістю `cost=0.00..1750.00`. Фактичний час сканування становив `0.005..18.837 ms`. Із 50 000 записів умові відповідали 3 334 рядки, а 46 666 були відкинуті (`Rows Removed by Filter`). Загальний `Execution Time` становив **19.788 ms**, а використання буферів — `shared hit=1000`.

Після створення GIN-індексу з `jsonb_path_ops` розміром **1144 kB** PostgreSQL змінив план виконання на **Bitmap Index Scan + Bitmap Heap Scan**. Оцінена вартість сканування таблиці зменшилася до `cost=35.97..1090.32`. `Bitmap Index Scan` знайшов 3 334 відповідні рядки за `2.446 ms`, після чого `Bitmap Heap Scan` отримав їх із таблиці. Загальний `Execution Time` зменшився до **18.003 ms**, тобто приблизно на **9%**. Водночас було прочитано 1000 блоків таблиці (`Heap Blocks: exact=1000`), а загальне використання буферів становило `shared hit=1004`.

Другий запит із фільтром `referrer = 'google'` також використав **Bitmap Index Scan + Bitmap Heap Scan**. Індекс знайшов 2 500 рядків за `0.354 ms`, а загальний час виконання становив **5.239 ms**. Це підтвердило, що створений GIN-індекс підтримує різні JSONB containment-запити з оператором `@>`.

### Step 4. Expression B-tree Index

Далі я перевірила запит `payload ->> 'device' = 'mobile'`. До створення B-tree індексу PostgreSQL використав **Seq Scan** з `cost=0.00..1750.00`. Запит знайшов 8 333 рядки, відкинув 41 667 і виконався за **11.097 ms**.

Після створення expression B-tree індексу PostgreSQL використав **Bitmap Index Scan + Bitmap Heap Scan**. Оцінена вартість сканування зменшилася до `cost=96.89..1221.91`, а `Bitmap Index Scan` знайшов 8 333 рядки за `2.980 ms`. Загальний `Execution Time` становив **6.349 ms**, що приблизно на **43% швидше**, ніж без індексу.

### Висновок

Експеримент показав, що ефективність індексації залежить від структури SQL-запиту та використаних операторів. GIN із `jsonb_path_ops` підходить для JSONB containment-запитів, тоді як expression B-tree індекс ефективний для текстових порівнянь конкретних JSONB-полів.

У моїх вимірюваннях обидва індекси покращили продуктивність, але GIN забезпечив лише помірне прискорення першого запиту, оскільки PostgreSQL усе ще звертався до всіх 1000 блоків таблиці. Також я побачила, що `cost` — це оцінка оптимізатора, а `actual time` та `Execution Time` — фактичні показники виконання. Тому вибір індексу потрібно перевіряти за реальними планами виконання, а не лише за його типом.

Task 5. JSONB vs. Normalized + Reflection

Step 1. Normalized table for checkout events

In [142]:
%%sql

DROP TABLE IF EXISTS t7_hw_checkouts_norm;

CREATE TABLE t7_hw_checkouts_norm (
    event_id        INTEGER PRIMARY KEY,
    user_id         INTEGER NOT NULL,
    event_ts        TIMESTAMPTZ NOT NULL,
    order_id        TEXT NOT NULL,
    total           NUMERIC(10, 2) NOT NULL,
    items_count     INTEGER NOT NULL CHECK (items_count > 0),
    payment_method  TEXT NOT NULL
                    CHECK (payment_method IN ('card', 'paypal', 'crypto'))
);

 * postgresql://postgres:***@/postgres?host=%2Fprivate%2Ftmp%2Fhw7_pg
Done.
Done.


[]

In [143]:
%%sql

INSERT INTO t7_hw_checkouts_norm (
    event_id,
    user_id,
    event_ts,
    order_id,
    total,
    items_count,
    payment_method
)
SELECT
    event_id,
    user_id,
    event_ts,
    payload ->> 'order_id' AS order_id,
    (payload ->> 'total')::NUMERIC(10, 2) AS total,
    (payload ->> 'items_count')::INTEGER AS items_count,
    payload ->> 'payment_method' AS payment_method
FROM t7_hw_events
WHERE event_type = 'checkout';

 * postgresql://postgres:***@/postgres?host=%2Fprivate%2Ftmp%2Fhw7_pg
10000 rows affected.


[]

In [144]:
%%sql

CREATE INDEX idx_t7_checkouts_norm_pm
    ON t7_hw_checkouts_norm (payment_method);

ANALYZE t7_hw_checkouts_norm;

 * postgresql://postgres:***@/postgres?host=%2Fprivate%2Ftmp%2Fhw7_pg
Done.
Done.


[]

In [145]:
%%sql

SELECT COUNT(*) AS n_checkouts
FROM t7_hw_checkouts_norm;

 * postgresql://postgres:***@/postgres?host=%2Fprivate%2Ftmp%2Fhw7_pg
1 rows affected.


n_checkouts
10000


2. JSONB version

In [146]:
%%sql

EXPLAIN (ANALYZE, BUFFERS)
SELECT
    payload ->> 'payment_method' AS payment_method,
    AVG((payload ->> 'total')::NUMERIC) AS avg_total,
    COUNT(*) AS n
FROM t7_hw_events
WHERE event_type = 'checkout'
GROUP BY payload ->> 'payment_method'
ORDER BY avg_total DESC
LIMIT 3;

 * postgresql://postgres:***@/postgres?host=%2Fprivate%2Ftmp%2Fhw7_pg
18 rows affected.


QUERY PLAN
Limit (cost=1955.65..1955.65 rows=3 width=72) (actual time=9.413..9.543 rows=3 loops=1)
Buffers: shared hit=1000
-> Sort (cost=1955.65..1969.55 rows=5560 width=72) (actual time=9.412..9.413 rows=3 loops=1)
Sort Key: (avg(((payload ->> 'total'::text))::numeric)) DESC
Sort Method: quicksort Memory: 25kB
Buffers: shared hit=1000
-> HashAggregate (cost=1800.39..1883.79 rows=5560 width=72) (actual time=8.062..8.081 rows=3 loops=1)
Group Key: (payload ->> 'payment_method'::text)
Batches: 1 Memory Usage: 217kB
Buffers: shared hit=1000


3. Normalized version

In [147]:
%%sql

EXPLAIN (ANALYZE, BUFFERS)
SELECT
    payment_method,
    AVG(total) AS avg_total,
    COUNT(*) AS n
FROM t7_hw_checkouts_norm
GROUP BY payment_method
ORDER BY avg_total DESC
LIMIT 3;

 * postgresql://postgres:***@/postgres?host=%2Fprivate%2Ftmp%2Fhw7_pg
16 rows affected.


QUERY PLAN
Limit (cost=269.06..269.07 rows=3 width=46) (actual time=3.162..3.164 rows=3 loops=1)
Buffers: shared hit=94
-> Sort (cost=269.06..269.07 rows=3 width=46) (actual time=3.161..3.162 rows=3 loops=1)
Sort Key: (avg(total)) DESC
Sort Method: quicksort Memory: 25kB
Buffers: shared hit=94
-> HashAggregate (cost=269.00..269.04 rows=3 width=46) (actual time=3.138..3.139 rows=3 loops=1)
Group Key: payment_method
Batches: 1 Memory Usage: 24kB
Buffers: shared hit=94


Step 3 (Bonus): We compare how to execute the same analytical query in PostgreSQL and MongoDB.

// MongoDB equivalent: avg total checkout per payment_method, top-3
// Pseudocode — не виконується у PostgreSQL

db.events.aggregate([
  {
    $match: {
      event_type: "checkout"
    }
  },
  {
    $group: {
      _id: "$payload.payment_method",
      avg_total: {
        $avg: {
          $toDouble: "$payload.total"
        }
      },
      n: { $sum: 1 }
    }
  },
  {
    $sort: {
      avg_total: -1
    }
  },
  {
    $limit: 3
  }
]);

## Task 5. Reflection — JSONB vs Normalized Table

У цьому завданні я порівняла продуктивність аналітичного запиту для двох способів зберігання checkout-подій: у JSONB та в нормалізованій SQL-таблиці. Для аналізу використовувала `EXPLAIN (ANALYZE, BUFFERS)`.

**JSONB-версія.** PostgreSQL використав `Seq Scan` із `cost=0.00..1650.06`, прочитав 50 000 подій, із яких 10 000 відповідали умові `event_type = 'checkout'`, а 40 000 були відкинуті. Фактичний час сканування становив `0.488..5.637 ms`. Далі `HashAggregate` згрупував записи за `payment_method` у 3 групи (`actual time=8.062..8.081 ms`), після чого виконалися `Sort` і `Limit`. Загальний `Execution Time` становив **10.704 ms**, використання буферів — `shared hit=1000`.

**Normalized-версія.** PostgreSQL також використав `Seq Scan`, але вже для таблиці з 10 000 checkout-записів. Оцінена вартість сканування була значно нижчою — `cost=0.00..194.00`, а фактичний час — `0.015..0.961 ms`. Оператор `HashAggregate` сформував 3 групи за `3.138..3.139 ms`, після чого виконалися `Sort` і `Limit`. Загальний `Execution Time` становив **3.235 ms**, використання буферів — лише `shared hit=94`.

Отже, у моєму середовищі нормалізована версія виконалася приблизно **у 3.3 раза швидше** (3.235 ms проти 10.704 ms), а час виконання зменшився на 69.8%. Це пояснюється тим, що нормалізована таблиця містить лише checkout-події, використовує типізовані колонки та не потребує витягування значень із JSONB і перетворення `total` на `NUMERIC`. Водночас експеримент не дозволяє відокремити вплив структури даних від різниці в розмірах таблиць.

Обидва запити використовували `Seq Scan`, `HashAggregate`, `Sort` і `Limit`, без `Index Scan`. Це показує, що для агрегації всіх записів послідовне сканування може бути ефективнішим за індексний пошук.

**Висновок.** PostgreSQL JSONB зручний для зберігання подій із різною структурою, особливо коли потрібні SQL-запити, `JOIN`, транзакції та інтеграція з нормалізованими таблицями. MongoDB Aggregation Pipeline є природним підходом для document-first workloads. Великі обсяги сирих clickstream-даних можуть бути зручнішими для зберігання в document store або data lake, залежно від вимог до масштабування й аналітики.

Для підготовлених ML features я б обрала типізовані SQL-таблиці, оскільки вони забезпечують контроль типів, перевірку якості даних і зручнішу побудову аналітичних запитів. Водночас JSONB залишається корисним для зберігання сирих подій перед їх подальшим перетворенням.

## Reflection — JSONB, GIN Indexes and Normalized Tables

У завданнях 4–5 я дослідила продуктивність JSONB-запитів, вплив індексів та переваги нормалізованих таблиць у PostgreSQL.

**1. Вплив GIN-індексу.** До створення індексу запит `payment_method = 'card'` використовував `Seq Scan` із `total cost=1750.00` та `Execution Time=19.788 ms`. Після створення GIN-індексу `jsonb_path_ops` план змінився на `Bitmap Index Scan` + `Bitmap Heap Scan`. Оцінена вартість сканування зменшилася до `1090.32`, а загальний час виконання — до `18.003 ms` (приблизно на 9%). Індекс знайшов 3334 рядки за `2.446 ms`.

**2. GIN vs B-tree.** GIN ефективний для оператора `@>`, оскільки індексує структуру та значення JSONB. Проте для текстового порівняння `payload ->> 'device' = 'mobile'` доцільніший expression B-tree index. У моєму експерименті він змінив `Seq Scan` на `Bitmap Index Scan` + `Bitmap Heap Scan` і скоротив `Execution Time` з `11.097 ms` до `6.349 ms` (на 42.8%).

**3. Read/write trade-off.** GIN прискорює пошук за JSONB, але потребує додаткового дискового простору (мій індекс займав 1144 kB). Операції `INSERT`, `UPDATE` і `DELETE` можуть стати дорожчими через необхідність підтримувати індекс.

**4. JSONB vs normalized для ML.** JSONB я обрала б для сирих подій із різною структурою, а нормалізовані таблиці — для підготовлених ML features із визначеними типами даних. У завданні 5 normalized-запит виконався за `3.235 ms` проти `10.704 ms` для JSONB. Обидва використовували `Seq Scan` і `HashAggregate`, але normalized-таблиця потребувала лише 94 буфери проти 1000.

**5. Додавання refund.** У JSONB-підході достатньо додати новий `event_type='refund'` із відповідним `payload`, оновити валідацію та обробку подій. Міграція схеми необов'язкова. У normalized-підході я створила б окрему таблицю `refunds` із типізованими колонками, обмеженнями та індексами, а також оновила ETL-процес.

Отже, вибір структури даних та індексів залежить від характеру запитів, вимог до продуктивності й подальшого використання даних у ML.